# Gridworld and value iteration

Consider an environment: 4x4 grid, cell numbers 0..15.

The agent can move: 0 = up, 1 = down, 2 = left, 3 = right.  If you move into a wall, you stay **put**

Terminal states: cell 0 (upper left corner) and cell 15 (lower right). Entering these states yields a reward of 0, and the episode ends (we can assume that there are no transitions from these states, or they are absorbing with a reward of 0).

Problem: Find the optimal values ​​of the states $V^*(s)$ and the optimal policy.

## Value iteration

### Deterministic

Transitions are deterministic.

Reward: -1 for each step (except terminal states).

In [20]:
import numpy as np

In [21]:
# Grid parameters
n_rows, n_cols = 4, 4
n_states = n_rows * n_cols
n_actions = 4  # 0: up, 1: down, 2: left, 3: right

# Step reward (for non-terminal states)
step_reward = -1.0
gamma = 1         # discount
theta = 1e-6         # convergence threshold

terminal_states = [0, 15]

# Value function
V = np.zeros(n_states)

In [22]:
def step(state, action):
    if state in terminal_states:
        return state, 0.0
    
    row = state // n_cols
    col = state % n_cols
    
    if action == 0:
        row = max(row-1, 0)
    elif action == 1:
        row = min(row+1, n_rows-1)
    elif action == 2:
        col = max(col-1, 0)
    elif action == 3:
        col = min(col+1, n_cols-1)
        
    next_state = row*n_cols + col
    reward = step_reward
    if next_state in terminal_states:
        reward = 0
    return next_state, reward

In [23]:
# Value iteration
while True:
    delta = 0
    for s in range(n_states):
        if s in terminal_states:
            continue
        v = V[s]
        values = []
        for a in range(n_actions):
            next_s, r = step(s, a)
            values.append(r + gamma * V[next_s])
        V[s] = max(values)
        delta = max(delta, abs(v-V[s]))
    if delta < theta:
        break
print('Optimal V=')
print(V.reshape(n_rows, n_cols))

Optimal V=
[[ 0.  0. -1. -2.]
 [ 0. -1. -2. -1.]
 [-1. -2. -1.  0.]
 [-2. -1.  0.  0.]]


In [24]:
# optimal policy
policy = np.zeros(n_states, dtype=int)
for s in range(s):
    if s in terminal_states:
        policy[s] = -1 # no action
        continue
    best_value = -np.inf
    best_action = 0
    for a in range(n_actions):
        next_s, r = step(s, a)
        v = r + gamma*V[next_s]
        if v>best_value:
            best_value = v
            best_action = a
    policy[s] = best_action

action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→', -1: 'T'}
policy_grid = np.array([action_symbols[a] for a in policy]).reshape(n_rows, n_cols)

print('Policy:')
print(policy_grid)

Policy:
[['T' '←' '←' '↓']
 ['↑' '↑' '↑' '↓']
 ['↑' '↑' '↓' '↓']
 ['↑' '→' '→' '↑']]


### Non-deterministic

In [25]:
p_intended = 0.8   # probability to perform selected direction
p_cw      = 0.1   # probability of clocwise rotetion by  90 degree
p_ccw     = 0.1   # probability of counter-clocwise rotetion by  90 degree

step_reward = -1.0
gamma = 0.9        
theta = 1e-6     

terminal_states = [0, 15]

action_directions = {
    0: [(-1, 0), (0, 1), (0, -1)],   # ↑, cw →, ccw ←
    1: [( 1, 0), (0,-1), (0,  1)],   # ↓, cw ←, ccw →
    2: [( 0,-1), (-1,0), ( 1,  0)],  # ←, cw ↑, ccw ↓
    3: [( 0, 1), ( 1,0), (-1,  0)]   # →, cw ↓, ccw ↑
}
 
probs = [p_intended, p_cw, p_ccw]

In [26]:
def step(state, action):
    if state in terminal_states:
        return []
    
    row = state // n_cols
    col = state % n_cols
    
    outcomes = {}   # key: next_state, value: {'prob': total probability, 'reward': reward}
    
    for (delta_r, delta_c), prob in zip(action_directions[action], probs):
        new_row = row + delta_r
        new_col = col + delta_c
        
        if new_row < 0 or new_row >= n_rows or new_col < 0 or new_col >= n_cols:
            new_row, new_col = row, col # return if out of grid
            
        next_state = new_row * n_cols + new_col
        if next_state in terminal_states:
            reward = 0
        else:
            reward = step_reward
            
        if next_state not in outcomes:
            outcomes[next_state] = {'prob': 0.0, 'reward': reward}
        outcomes[next_state]['prob'] += prob
        
    return [(data['prob'], next_s, data['reward']) for next_s, data in outcomes.items()]        

In [27]:
V = np.zeros(n_states)
while True:
    delta = 0.0
    for s in range(n_states):
        if s in terminal_states:
            continue
        
        old_v = V[s]
        best_value = -np.inf
        for a in range(n_actions):
            expected_value = 0
            for prob, next_s, r in step(s, a):
                expected_value += prob * (r + gamma * V[next_s])
            if expected_value > best_value:
                best_value = expected_value
        V[s] = best_value
        delta = max(delta, abs(old_v - V[s]))
        
    if delta < theta:
        break
    
print('Optimal V=')
print(np.round(V.reshape(n_rows, n_cols), 3))

Optimal V=
[[ 0.    -0.369 -1.626 -2.546]
 [-0.369 -1.513 -2.372 -1.626]
 [-1.626 -2.372 -1.513 -0.369]
 [-2.546 -1.626 -0.369  0.   ]]


In [28]:
policy = np.full(n_states, -1, dtype=int)  # -1 for terminal states
action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→', -1: 'T'}

for s in range(n_states):
    if s in terminal_states:
        continue
    best_value = -np.inf
    best_a = 0
    for a in range(n_actions):
        expected_value = 0
        for prob, next_s, r in step(s, a):
            expected_value += prob * (r + V[next_s])
        if expected_value > best_value:
            best_value = expected_value
            best_a = a
    policy[s] = best_a
policy_grid = np.array([action_symbols[a] for a in policy]).reshape(n_rows, n_cols)

print('Policy:')
print(policy_grid)


Policy:
[['T' '←' '←' '←']
 ['↑' '↑' '←' '↓']
 ['↑' '↑' '↓' '↓']
 ['↑' '→' '→' 'T']]


## Policy iteration

### Deterministic

In [29]:
n_rows, n_cols = 4, 4
n_states = n_rows * n_cols
n_actions = 4  # 0: ↑, 1: ↓, 2: ←, 3: →

step_reward = -1.0
gamma = 0.9
theta = 1e-6  # convergence threshold for policy

terminal_states = {0, 15}

In [30]:
def step(state, action):
    if state in terminal_states:
        return state, 0.0
    
    row = state // n_cols
    col = state % n_cols
    
    if action == 0:
        row = max(row-1, 0)
    elif action == 1:
        row = min(row+1, n_rows-1)
    elif action == 2:
        col = max(col-1, 0)
    elif action == 3:
        col = min(col+1, n_cols-1)
        
    next_s = row*n_cols + col
    reward = 0.0 if next_s in terminal_states else step_reward
    return next_s, reward

In [31]:
def policy_evaluation(policy, V, gamma, theta):
    while True:
        delta = 0
        for s in range(n_states):
            if s in terminal_states:
                continue
            v = V[s]
            a = policy[s]
            next_s, r = step(s, a)
            V[s] = r + gamma * V[next_s]
            delta = max(delta, abs(v-V[s]))
        if delta < theta:
            break
    return V

In [32]:
def policy_improvement(policy, V):
    policy_stable = True
    new_policy = np.full(n_states, -1, dtype=int)
    for s in range(n_states):
        if s in terminal_states:
            continue
        best_value = -np.inf
        best_action = 0
        for a in range(n_actions):
            next_s, r = step(s, a)
            val = r + gamma * V[next_s]
            if val > best_value:
                best_value = val
                best_action = a
        new_policy[s] = best_action
        if policy[s] != best_action:
            policy_stable = False
    return new_policy, policy_stable

In [33]:
policy = np.full(n_states, 0, dtype=int)
for s in terminal_states:
    policy[s] = -1
    
V = np.zeros(n_states)

iteration = 0
while True:
    iteration += 1
    V = policy_evaluation(policy, V, gamma, theta)
    new_policy, policy_stable = policy_improvement(policy, V)
    if policy_stable:
        policy = new_policy
        break
    policy = new_policy
    
print(f"Numner of iterations: {iteration}, Optimal V*=")
print(np.round(V.reshape(n_rows, n_cols), 3))

action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→', -1: 'T'}
policy_grid = np.array([action_symbols[a] for a in policy]).reshape(n_rows, n_cols)
print("\nOptimal policy:")
print(policy_grid)

Numner of iterations: 4, Optimal V*=
[[ 0.   0.  -1.  -1.9]
 [ 0.  -1.  -1.9 -1. ]
 [-1.  -1.9 -1.   0. ]
 [-1.9 -1.   0.   0. ]]

Optimal policy:
[['T' '←' '←' '↓']
 ['↑' '↑' '↑' '↓']
 ['↑' '↑' '↓' '↓']
 ['↑' '→' '→' 'T']]


### Non-deterministic

In [34]:
p_intended = 0.8
p_cw = 0.1
p_ccw = 0.1

step_reward = -1.0
gamma = 0.9
theta = 1e-6

terminal_states = {0, 15}

action_directions = {
    0: [(-1, 0), (0, 1), (0, -1)],   # ↑, cw →, ccw ←
    1: [( 1, 0), (0,-1), (0,  1)],   # ↓, cw ←, ccw →
    2: [( 0,-1), (-1,0), ( 1,  0)],  # ←, cw ↑, ccw ↓
    3: [( 0, 1), ( 1,0), (-1,  0)]   # →, cw ↓, ccw ↑
}

probs = [p_intended, p_cw, p_ccw]

In [35]:
def step(state, action):
    if state in terminal_states:
        return []
    
    row = state // n_cols
    col = state % n_cols
    
    outcomes = {}
    
    for (delta_r, delta_c), prob in zip(action_directions[action], probs):
        new_row = row + delta_r
        new_col = col + delta_c
        if new_row < 0 or new_row >= n_rows or new_col < 0 or new_col >= n_cols:
            new_row, new_col = row, col
        next_state = new_row * n_cols + new_col
        reward = 0.0 if next_state in terminal_states else step_reward
        
        if next_state not in outcomes:
            outcomes[next_state] = {'prob': 0.0, 'reward': reward}
        outcomes[next_state]['prob'] += prob
    return [(data['prob'], next_s, data['reward']) for next_s, data in outcomes.items()]

In [36]:
def policy_evaluation(policy, V, gamma, theta):
    while True:
        delta = 0
        for s in range(n_states):
            if s in terminal_states:
                continue
            v = V[s]
            a = policy[s]
            expected_value = 0
            for prob, next_s, r in step(s, a):
                expected_value += prob * (r + gamma * V[next_s])
            V[s] = expected_value
            delta = max(delta, abs(v-V[s]))
        if delta < theta:
            break
    return V

In [37]:
def policy_improvement(policy, V, gamma):
    policy_stable =True
    new_policy = np.full(n_states, -1, dtype=int)
    for s in range(n_states):
        if s in terminal_states:
            continue
        best_value = -np.inf
        best_action = 0
        for a in range(n_actions):
            expected_value = 0
            for prob, next_s, r in step(s, a):
               expected_value += prob * (r + gamma * V[next_s])
            if expected_value > best_value:
                best_value = expected_value
                best_action = a
        new_policy[s] = best_action
        if policy[s] != best_action:
            policy_stable = False
    return new_policy, policy_stable

In [38]:
policy = np.full(n_states, 0, dtype=int)  
for s in terminal_states:
    policy[s] = -1
    
V = np.zeros(n_states)

iteration = 0
while True:
    iteration += 1
    V = policy_evaluation(policy, V, gamma, theta)
    new_policy, policy_stable = policy_improvement(policy, V, gamma)
    if policy_stable:
        policy = new_policy
        break
    policy = new_policy
    
print(f"Number of iterations: {iteration}, Optimal V*=")
print(np.round(V.reshape(n_rows, n_cols), 3))

action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→', -1: 'T'}
policy_grid = np.array([action_symbols[a] for a in policy]).reshape(n_rows, n_cols)
print("\nOptimal policy:")
print(policy_grid)

Number of iterations: 4, Optimal V*=
[[ 0.    -0.369 -1.626 -2.546]
 [-0.369 -1.513 -2.372 -1.626]
 [-1.626 -2.372 -1.513 -0.369]
 [-2.546 -1.626 -0.369  0.   ]]

Optimal policy:
[['T' '←' '←' '↓']
 ['↑' '↑' '↓' '↓']
 ['↑' '→' '→' '↓']
 ['→' '→' '→' 'T']]
